# 워크숍 1 — 기초

이 노트북에서는 Menlo Robotics 플랫폼에 연결하고, 시뮬레이션 로봇을 만든 뒤 **menlo_robot_sdk**로 이동 명령을 보내는 방법을 단계별로 배웁니다.

이 튜토리얼을 마치면 다음을 할 수 있습니다.
1. API 키로 Menlo 플랫폼 인증하기
2. 가상 로봇을 만들고 연결하기
3. 브라우저에서 로봇을 볼 수 있는 3D 뷰어 열기
4. 로봇이 사용할 수 있는 스킬 확인하기
5. `set_velocity`와 `go_to` 명령으로 로봇 움직이기
6. 로봇의 1인칭 카메라 이미지 캡처하기
7. 장애물 주변을 이동하는 간단한 명령 체인 작성하기

## 준비 사항

시작하기 전에 다음을 준비하세요.

- **Google 계정**(Google Colab 사용)
- Menlo API 키([console.menlo.ai](https://console.menlo.ai) → Settings → API Keys에서 발급)
- **Google Chrome**(3D 뷰어는 Chrome에서 가장 안정적으로 작동합니다)

## API 키 설정하기

이 노트북은 두 가지 실행 방식을 지원합니다. 자신의 환경에 맞는 방식을 선택하세요.

### 모드 A: Google Colab(Secrets 관리자)

Colab의 Secrets 관리자는 키를 안전하게 저장하며 세션이 바뀌어도 유지합니다.

1. Colab 왼쪽 사이드바에서 **열쇠 아이콘**(🔑)을 클릭합니다.
2. **"Add new secret"**을 누르고 아래 키를 각각 추가합니다.

| Secret 이름 | 값 |
|---|---|
| `MENLO_API_KEY` | Menlo API 키([console.menlo.ai](https://console.menlo.ai) → Settings → API Keys) |
| `TOKAMAK_API_KEY` | Tokamak API 키(섹션 7의 VLM에서 사용) |

3. 각 Secret의 **"Notebook access"**를 켭니다.

> **참고:** 노트북을 공유해도 Secret은 포함되지 않습니다. 각 사용자가 자신의 키를 직접 설정해야 합니다.

### 모드 B: 로컬 / `.env` 파일

VS Code나 JupyterLab 등 로컬 환경에서 실행한다면 노트북과 같은 폴더에 `.env` 파일을 만드세요.

```
MENLO_API_KEY=sk_live_abc123...
TOKAMAK_API_KEY=tok_abc123...
```

다음 셀은 현재 환경을 자동으로 감지하여 알맞은 방식으로 키를 불러옵니다.

---

# 섹션 4: SDK 기본 사용법

## SDK의 주요 import

`menlo_robot_sdk` 패키지는 다음과 같은 핵심 기능을 제공합니다.

| Import | 역할 |
|---|---|
| `AsyncClient` | Menlo 플랫폼과 통신하는 HTTP 클라이언트(로봇 생성, 세션 관리 등) |
| `connect` | 로봇과 실시간 세션을 열고 `MenloSession`을 반환하는 고수준 함수 |
| `generate_room_key` | *(실험 기능)* 브라우저 기반 3D 뷰어용 일회성 키 생성 |

연결 후에는 **`session`** 객체를 주로 사용합니다.
- `session.discover_skills()` — 로봇이 할 수 있는 작업 목록 확인
- `session.invoke(skill, params)` — 스킬을 실행하고 결과가 올 때까지 대기
- `session.get_vision(camera)` — 카메라 프레임 가져오기
- `session.state.get(key)` — 현재 로봇 상태 읽기
- `session.disconnect()` — 연결 종료

**`robot_id`**는 플랫폼에서 각 로봇을 구분하는 고유 식별자입니다. `client.robots.create()`를 호출하면 반환됩니다.

## 1단계: 의존성 설치

Colab에는 `menlo_robot_sdk`가 기본 설치되어 있지 않으므로 여기서 설치합니다. Colab 런타임이 초기화되면 패키지도 사라지므로 세션마다 한 번 실행해야 합니다.

In [ ]:
!pip install -q "menlo-robot-sdk[livekit]"

## 2단계: 설정 불러오기

`google.colab.userdata`를 사용해 Colab Secrets 관리자에서 API 키를 불러옵니다.

In [ ]:
import os
import time
import asyncio

# ── 키 로더: 먼저 .env를 확인한 다음 Colab Secrets를 확인합니다 ────────────────────────
def _load_keys():
    # 모드 B: 로컬 .env 파일
    try:
        from dotenv import load_dotenv
        load_dotenv(override=False)
        if os.environ.get("MENLO_API_KEY"):
            print("Loaded keys from .env file")
            return "dotenv"
    except ImportError:
        pass  # python-dotenv not installed → not in .env mode

    # 모드 A: Google Colab Secrets
    try:
        from google.colab import userdata
        os.environ.setdefault("MENLO_API_KEY",    userdata.get("MENLO_API_KEY") or "")
        os.environ.setdefault("TOKAMAK_API_KEY",  userdata.get("TOKAMAK_API_KEY") or "")
        print("Loaded keys from Colab Secrets")
        return "colab"
    except Exception:
        pass

    return "env"  # keys already in environment (CI, Docker, etc.)

_load_keys()

MENLO_API_KEY   = os.environ.get("MENLO_API_KEY", "")
TOKAMAK_API_KEY = os.environ.get("TOKAMAK_API_KEY", "")
RCS_URL         = "https://platform-auth.menlo.ai/rcs"
VIEWER_BASE_URL = "https://sim.menlo.ai"

assert MENLO_API_KEY and not MENLO_API_KEY.startswith("sk_live_YOUR"), (
    "MENLO_API_KEY not set!\n"
    "  • Colab: add it in the Secrets manager (key icon in the left sidebar)\n"
    "  • Local: add MENLO_API_KEY=... to a .env file next to this notebook"
)
print(f"Configuration loaded — platform: {RCS_URL}")
print(f"MENLO_API_KEY    : {MENLO_API_KEY[:12]}...")
print(f"TOKAMAK_API_KEY  : {'(set)' if TOKAMAK_API_KEY else '(not set — needed for Section 7)'}")

## 잠깐 살펴보기: `await`는 무슨 뜻일까요?

이 튜토리얼의 SDK 호출은 대부분 `await`로 시작합니다. 처음 보는 학생을 위해 핵심만 살펴보겠습니다.

**문제:** 인터넷을 통해 로봇과 통신하려면 시간이 걸립니다. 로봇 생성을 요청하거나 이동 명령을 보냈을 때 결과가 즉시 오지 않고 몇 밀리초에서 몇 초까지 걸릴 수 있습니다. 일반적인 Python 코드는 결과를 기다리는 동안 멈춰 있게 됩니다.

**해결:** Python의 *async*(비동기) 함수는 네트워크 응답처럼 느린 작업을 기다리는 동안 전체 프로그램을 막지 않고 잠시 멈출 수 있습니다. `await`는 다음 뜻입니다.

> *"이 작업을 시작하고, 끝날 때까지 여기서 기다린 다음 결과를 돌려줘."*

```python
# 일반 함수 호출 — 즉시 반환합니다.
result = some_function()

# 비동기 호출 — 작업을 시작하고 완료될 때까지 기다립니다.
result = await some_async_function()
```

**기억할 규칙:** 플랫폼이나 로봇과 통신하는 모든 `menlo_robot_sdk` 호출 앞에는 `await`가 필요합니다. 빠뜨리면 데이터 대신 `<coroutine object ...>` 같은 값이 나오거나 `RuntimeWarning: coroutine was never awaited` 경고가 표시됩니다. 이 경고는 거의 항상 `await`를 빠뜨렸다는 뜻입니다.

**노트북 셀에서는 왜 `await`를 바로 쓸 수 있을까요?** 일반 Python에서는 `async def` 함수 안에서만 사용할 수 있지만, Jupyter와 Colab의 셀은 이미 *이벤트 루프* 안에서 실행되므로 최상위에서도 사용할 수 있습니다. 이 코드를 일반 `.py` 파일로 옮긴다면 `async def main(): ...`으로 감싸고 `asyncio.run(main())`으로 실행해야 합니다.

## 3단계: 클라이언트와 로봇 만들기

`AsyncClient`는 RCS URL과 API 키를 사용해 Menlo 플랫폼에 연결합니다. 그런 다음 `asimov-v0` 창고 모델의 시뮬레이션 로봇을 만듭니다.

In [ ]:
from menlo_robot_sdk import AsyncClient

client = AsyncClient(rcs_url=RCS_URL, api_key=MENLO_API_KEY)

created = await client.robots.create(name=f"sdk-test-{int(time.time())}", model="asimov-v0")
robot_id = created.robot.id
print(f"Created robot: {robot_id}")

## 4단계: 세션 시작하기

세션을 열면 노트북이 로봇의 방에 참여하여 명령을 보낼 수 있습니다. 직접 연결을 끊을 때까지 세션은 열린 상태로 유지됩니다.

In [ ]:
from menlo_robot_sdk import connect

session = await connect(
    client,
    robot_id,
    worker_names=[],                  # no server-side worker — the browser is the runtime
    rcw_identity_prefix="simplesim",
    join_livekit=True,
)
print(f"Connected! room = {session.session.room_name}")

## 5단계: 3D 뷰어 열기

SDK가 일회성 뷰어 키를 생성하므로 브라우저에서 로봇을 볼 수 있습니다. **출력된 URL을 Chrome의 새 탭에서 열고**, 3D 창고 장면이 완전히 로드될 때까지 기다린 뒤 계속하세요.

뷰어 링크는 4시간 동안 유효하며 회의 링크처럼 다른 사람과 공유할 수 있습니다.

In [ ]:
from menlo_robot_sdk.experimental import generate_room_key

room_key = await generate_room_key(client, robot_id)
viewer_url = f"{VIEWER_BASE_URL}/?key={room_key}"

print("=" * 60)
print(f"VIEWER URL: {viewer_url}")
print("=" * 60)
print()
print("1. Open the URL above in a new Chrome tab")
print("2. Wait for the 3D warehouse robot to load")
print("3. Then run the next cell")

## 6단계: 스킬이 준비될 때까지 기다리기

스킬은 로봇이 수행할 수 있는 동작입니다. 시뮬레이션은 브라우저에서 실행되므로 스킬을 찾으려면 뷰어가 열려 있어야 합니다.

이 셀은 Chrome 탭이 방에 참여할 때까지 계속 재시도합니다.

In [ ]:
async def wait_for_skills(timeout_s: float = 180.0):
    """뷰어가 접속하고 스킬을 사용할 수 있을 때까지 주기적으로 확인합니다."""
    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        try:
            found = await session.discover_skills()
            if found:
                return found
        except (RuntimeError, TimeoutError):
            pass  # viewer not joined yet
        await asyncio.sleep(2.0)
    raise TimeoutError("Viewer did not join — is the Chrome tab open?")


skills = await wait_for_skills()
print("Skills found:")
for skill in skills:
    print(f"  - {skill.name}")
    print(skill)

## 7단계: 로봇 상태 읽기

명령을 보내기 전에 현재 로봇 상태를 확인해 봅시다. `robot_status`에는 로봇의 위치, 방향과 전체 상태가 들어 있습니다.

In [ ]:
state = await session.state.get("robot_status")
print(f"Robot status : {state.robot.status}")
print(f"Robot entity : {state.robot.entity_id}")
if state.robot.pose:
    print(f"Position     : {state.robot.pose.position}")
    print(f"Yaw (deg)    : {state.robot.pose.yaw_deg}")
print(f"Runtime ready: {state.runtime.status}")
print(f"Accepts cmds : {state.runtime.accepts_tool_calls}")

## 8단계: 장면 상태 읽기

로봇 자체의 상태뿐 아니라 창고 안의 모든 엔티티(패드, 큐브, 로봇)와 위치 및 속성이 담긴 **장면 상태**도 읽을 수 있습니다.

장면 상태는 실제 로봇에서는 사용할 수 없는 시뮬레이션 전용 기능입니다. 미래에 로봇이 가질 수 있는 기능을 모의하기 위해 제공합니다. 스크립트가 장면 상태 정보에 의존하면 실제 로봇으로 그대로 옮길 수 없다는 점을 기억하세요.

장면에는 세 종류의 엔티티가 있습니다.

| 엔티티 종류 | 예 | 설명 |
|---|---|---|
| **로봇** | `robot` | 휴머노이드 자체의 위치, 방향, 들고 있는 물체 |
| **패드** | `pad_A`, `pad_B`, ... | 창고 바닥의 구역. `pad_A`는 큐브를 공급하는 컨베이어이며 나머지는 색상별 배송 패드입니다. |
| **큐브** | `cube_0`, `cube_1`, ... | 컨베이어 위에서 집고 배송할 수 있는 색깔 물체 |

멀리 (+20, +20)에 놓인 `visible=False` 상태의 `cube_pool_*` 엔티티도 보일 수 있습니다. 이는 시뮬레이터의 예비 큐브입니다. 큐브를 치우면 벨트가 이 숨겨진 풀에서 새 큐브를 가져오므로 창고의 큐브가 떨어지지 않습니다. `visible`로 필터링하면 숨길 수 있습니다.

In [ ]:
scene = await session.state.get("scene_state")

# 엔티티를 종류별로 분류합니다
robot_entities = []
pad_entities = []
cube_entities = []

for eid, entity in scene.entities.items():
    if eid == "robot":
        robot_entities.append((eid, entity))
    elif eid.startswith("pad_"):
        pad_entities.append((eid, entity))
    elif eid.startswith("cube") and entity.visible:
        cube_entities.append((eid, entity))
    # 보이지 않는 예비 큐브와 중복 별칭은 건너뜁니다

# --- 로봇 ---
print("=== Robot ===")
for eid, e in robot_entities:
    pos = e.pose.position
    print(f"  {eid:12s}  pos=({pos[0]:+.2f}, {pos[1]:+.2f}, {pos[2]:+.2f})  yaw={e.pose.yaw_deg:.1f}\u00b0")

# --- 패드 ---
print(f"\n=== Pads ({len(pad_entities)}) ===")
for eid, e in sorted(pad_entities):
    pos = e.pose.position
    print(f"  {eid:12s}  pos=({pos[0]:+.2f}, {pos[1]:+.2f}, {pos[2]:+.2f})")

# --- 보이는 큐브 ---
print(f"\n=== Cubes ({len(cube_entities)}) ===")
for eid, e in sorted(cube_entities):
    pos = e.pose.position
    color = e.state.get("color", "?") if e.state else "?"
    pad = e.state.get("parent_pad_id", "?") if e.state else "?"
    held = f"  [held by {e.attached_to}]" if e.attached_to else ""
    print(f"  {eid:12s}  color={color:6s}  on={pad:5s}  pos=({pos[0]:+.2f}, {pos[1]:+.2f}, {pos[2]:+.2f}){held}")

---

# 섹션 5: 휴머노이드 정책 뷰어 기본 사용법

## 뷰어 열기

아직 열지 않았다면 5단계에서 출력된 뷰어 URL을 Chrome 새 탭에서 여세요. 플랫폼 위에 휴머노이드 로봇이 서 있는 3D 창고 장면이 나타납니다.

시뮬레이션은 실제로 뷰어에서 실행됩니다. 노트북은 네트워크로 명령을 보내고 브라우저가 실시간으로 실행합니다.

## 카메라 조작

3D 뷰어에서 카메라를 움직여 다양한 각도에서 장면을 볼 수 있습니다.

| 동작 | 방법 |
|---|---|
| **회전** | 마우스 왼쪽 버튼으로 클릭한 채 드래그 |
| **이동** | 오른쪽 클릭 후 드래그 또는 Shift + 클릭 후 드래그 |
| **확대/축소** | 마우스 휠 |
| **카메라 초기화** | `R`을 눌러 기본 시점으로 복귀 |

지금 카메라를 움직여 창고 구조에 익숙해져 보세요. 로봇을 효율적으로 이동시키려면 공간 구조를 이해해야 합니다.

## Asimov 정책 뷰어

뷰어는 단순한 카메라가 아니라 `asimov-v0` 정책의 **런타임**입니다.

- 로봇 물리, 충돌 감지, 탐색이 모두 **브라우저 탭 안에서** 실행됩니다.
- Chrome 탭을 닫으면 시뮬레이션이 멈추고 SDK와 런타임의 연결도 끊어집니다.
- SDK 없이도 로봇을 직접 제어할 수 있는 여러 상호작용 모드를 제공합니다.

### 클릭하여 걷기

**지금 해보기:** 창고 바닥을 클릭하세요. 로봇이 클릭한 위치로 걸어갑니다. 내부적으로는 SDK에서 사용할 것과 동일한 `go_to` 스킬을 실행합니다.

### 자동 탐색

**지금 해보기:** 이름표가 붙은 패드(예: `pad_A`, `pad_B`, `pad_C`)를 클릭하세요. 로봇이 장애물을 피해 경로를 계획하고 걸어갑니다. 이 경로 탐색 덕분에 `go_to`는 단순 속도 명령보다 강력합니다.

### 수동 속도 명령

**지금 해보기:** 방향키로 로봇을 직접 움직이세요.

| 키 | 동작 |
|---|---|
| `W` / `Up` | 앞으로 걷기 |
| `S` / `Down` | 뒤로 걷기 |
| `A` / `Left` | 왼쪽으로 돌기 |
| `D` / `Right` | 오른쪽으로 돌기 |

이는 `set_velocity` SDK 명령의 수동 버전이며 키를 누르는 동안 로봇이 움직입니다.

## 로봇이 넘어졌는지 어떻게 알 수 있을까요?

뷰어에서는 옆이나 뒤로 누운 모습이 바로 보이지만 SDK에서는 코드로 확인해야 합니다. `robot_status`의 `status` 필드는 다음 값을 가질 수 있습니다.

| 상태 | 의미 |
|---|---|
| `ready` | 명령을 기다리는 중 |
| `moving` | 이동 동작 실행 중 |
| `holding` | 물체를 들고 있음 |
| **`fallen`** | **로봇이 넘어짐** |
| `error` | 오류 발생 |
| `unknown` | 아직 상태가 결정되지 않음 |

아래 셀을 실행해 확인하세요. `fallen`이면 맨 아래 문제 해결 안내에 따라 장면을 초기화해야 합니다.

In [ ]:
state = await session.state.get("robot_status")
print(f"Robot status: {state.robot.status}")

if state.robot.status == "fallen":
    print("\nThe robot has fallen over! You may need to reset the scene.")
else:
    print("\nRobot is upright and operational.")

---

# 섹션 6: Menlo SDK로 간단한 명령 체인 작성하기

이 섹션에서는 여러 SDK 명령을 이어 실행하는 짧은 프로그램을 작성합니다. 이런 방식으로 더 복잡한 로봇 행동을 만들 수 있습니다.

먼저 스크린샷을 노트북 안에 표시하는 도우미 함수를 사용합니다.

In [ ]:
from IPython.display import Image, display

async def screenshot(label: str = ""):
    """Grab the robot's POV camera and display it in the notebook."""
    jpeg = await session.get_vision("pov")
    if label:
        print(label)
    display(Image(data=jpeg, format="jpeg"))

## 연습 문제 1: 간단히 걷기

스크린샷을 찍고 로봇을 앞으로 움직인 다음 다시 스크린샷을 찍어 봅시다. 로봇의 시점에서 세상이 어떻게 보이고 이동에 따라 어떻게 달라지는지 확인할 수 있습니다.

In [ ]:
# 1. 이동 전에 스크린샷을 찍습니다
await screenshot("BEFORE — robot's point of view:")

In [ ]:
# 2. 2초 동안 앞으로 이동합니다
result = await session.invoke("set_velocity", {"vx": 0.8, "duration_s": 2.0})
print(f"set_velocity -> {result.status}")

In [ ]:
# 3. 이동 후 스크린샷을 찍습니다
await screenshot("AFTER — robot's point of view:")

### 생각해 볼 질문

위의 두 스크린샷을 살펴보고 다음 질문을 생각해 보세요.

1. **화면이 달라졌나요?** 이동 전후 이미지의 차이는 무엇인가요?
2. **왜 달라졌나요?** 로봇이 얼마나 이동했는지 추정할 수 있나요? (속력 × 시간 = 거리)
3. **로봇의 시점과 지금까지 본 시점은 어떻게 다른가요?** Chrome의 3D 뷰어와 로봇 POV 카메라에서 각각 볼 수 있는 것과 없는 것은 무엇인가요?
4. **로봇 센서의 한계 때문에 어떤 어려움이 생길까요?** POV 카메라만 사용할 수 있다면 어떤 작업이 더 어려워질까요?

## 연습 문제 2를 시작하기 전에: 원하는 각도만큼 회전하기

연습 문제 2에서는 로봇을 약 90° 돌려야 합니다. `set_velocity`에는 "90도 회전" 명령이 없고 *회전 속도*와 *지속 시간*만 받으므로 간단한 계산이 필요합니다.

### 각도가 아니라 라디안

회전 속도 `wz`의 단위는 **rad/s**입니다. 90°=π/2≈1.57, 180°=π≈3.14, 360°=2π≈6.28입니다.

**회전 각도(rad) = `wz` × `duration_s`**

### 특징 1: 자전거처럼 회전합니다

이 보행 정책은 제자리 회전을 할 수 없습니다. 자전거처럼 앞으로 움직이면서만 방향을 바꿀 수 있으므로 `wz`와 작은 `vx`를 함께 사용하세요.

```python
{"wz": 0.5, "vx": 0.2, "duration_s": 3.14}   # 정상적으로 회전
{"wz": 0.5, "duration_s": 3.14}              # 거의 회전하지 않음
```

작은 `vx` 때문에 제자리에서 돌지 않고 완만한 호를 그리므로 주변 공간도 고려해야 합니다.

### 특징 2: 속도 제한(알림 없이 잘림)

정책의 한계를 넘는 값은 별도 경고 없이 잘립니다. `vx`와 `vy`는 ±1.5 m/s, `wz`는 ±0.6 rad/s가 한계입니다. 예를 들어 `wz=1.0`을 요청해도 실제로는 0.6으로 회전하므로 계산한 각도와 달라집니다.

### 재사용 가능한 예시

```python
# 왼쪽으로 약 90° 회전: 0.5 rad/s × 3.14 s ≈ 1.57 rad
await session.invoke("set_velocity", {"wz": 0.5, "vx": 0.2, "duration_s": 3.14})
# 오른쪽으로 약 90° 회전: 음수 wz
await session.invoke("set_velocity", {"wz": -0.5, "vx": 0.2, "duration_s": 3.14})
# 왼쪽으로 약 180° 회전: 같은 속도로 두 배의 시간
await session.invoke("set_velocity", {"wz": 0.5, "vx": 0.2, "duration_s": 6.28})
```

### 오차를 예상하세요

계산값은 목표일 뿐 보장값이 아닙니다. 실제 시뮬레이터에서 90° 명령은 정책의 가속, 보행 흔들림, 감속 때문에 약 75~100°가 될 수 있습니다. 이후 `robot_status`의 `yaw_deg`로 실제 방향을 확인하세요. 결과를 확인하고 보정하는 것은 로봇 공학의 핵심 개념입니다.

## 연습 문제 2: 벽 돌아가기

Chrome 뷰어의 3인칭 화면에서 벽이나 장애물을 찾으세요. 시작 지점 앞의 컨베이어 벨트가 좋은 예입니다.

**과제:** `go_to` 없이 `set_velocity` 명령만 사용해 장애물 주변을 이동하는 스크립트를 작성하세요. `go_to`의 내장 경로 계획 없이 로봇을 제어하는 어려움을 경험하는 것이 목적입니다.

`vx`는 전후 속도(±1.5 m/s), `vy`는 좌우 이동 속도(±1.5 m/s), `wz`는 회전 속도(±0.6 rad/s)입니다. 회전할 때는 작은 `vx`도 함께 사용해야 합니다.

기본 전략은 다음과 같습니다.
1. 벽을 따라 바라보도록 약 90° 회전
2. 벽 끝을 지날 때까지 전진
3. 반대 방향으로 약 90° 회전
4. 전진하여 벽 너머로 이동
5. 필요하면 다시 회전하고 걸어 벽 뒤쪽에 도착

회전 대신 `vy`로 벽을 따라 옆걸음해도 됩니다. 명령 사이마다 스크린샷과 로봇 위치를 확인하세요. 아래 뼈대 코드의 값을 채워 보세요.

In [ ]:
# 여러 번 사용할 도우미: 로봇의 위치와 방향을 출력합니다
async def print_position(label: str = ""):
    state = await session.state.get("robot_status")
    p = state.robot.pose.position
    print(f"{label:20s} pos=({p[0]:+.2f}, {p[1]:+.2f})  yaw={state.robot.pose.yaw_deg:+.1f}°  status={state.robot.status}")

# 현재 위치를 확인하도록 스크린샷을 찍습니다
await screenshot("Starting position:")
await print_position("START")

In [ ]:
# ============================================================
# 여기에 코드를 작성하세요
# set_velocity 명령을 차례로 작성하여
# 로봇이 벽을 돌아가게 하세요.
#
# 명령 예시:
#   await session.invoke("set_velocity", {"vx": 0.8, "duration_s": 3.0})               # 약 2m 전진
#   await session.invoke("set_velocity", {"wz": 0.5, "vx": 0.2, "duration_s": 3.14})   # 왼쪽으로 약 90° 회전
#   await session.invoke("set_velocity", {"wz": -0.5, "vx": 0.2, "duration_s": 3.14})  # 오른쪽으로 약 90° 회전
#
# 기억하세요:
#   * 회전에는 작은 vx가 필요합니다. 이 로봇은 제자리 회전을 할 수 없습니다!
#   * |vx|, |vy| ≤ 1.5 and |wz| ≤ 0.6 — 더 큰 값은 알림 없이 제한됩니다
#
# 팁: 이동 사이에 진행 상황을 확인하세요.
#   await screenshot("회전 후:")
#   await print_position("회전 후")
# ============================================================



In [ ]:
# 최종 위치를 확인합니다
await screenshot("Final position:")

state = await session.state.get("robot_status")
print(f"Final position: {state.robot.pose.position}")
print(f"Final yaw: {state.robot.pose.yaw_deg} degrees")
print(f"Robot status: {state.robot.status}")

### 생각해 볼 질문

1. **스크립트가 매번 똑같이 작동하나요?** 2~3회 실행했을 때 정확히 같은 위치에 도착하는지, 그렇지 않다면 이유가 무엇인지 생각해 보세요.
2. **벽의 위치가 0.5m 정도 바뀌면 어떻게 될까요?** 이 제어 방식의 어떤 한계가 드러나나요?
3. **움직인 벽에도 대응하려면 어떤 정보가 필요하고 어디서 얻을 수 있을까요?** 지금까지 사용한 도구 중 로봇의 위치나 앞의 물체를 알려 주는 것이 있나요?

---

# 섹션 7: 집기와 놓기

창고 로봇의 핵심은 물체를 옮기는 일입니다. 이 섹션에서는 컨베이어의 큐브를 집어 알맞은 패드로 배송하는 고전적인 **pick and place** 작업을 수행합니다.

### 분류 게임

컨베이어(`pad_A`)가 색깔 큐브를 공급하며 각 배송 패드는 한 색만 받습니다.

| 색상 | 배송 패드 |
|---|---|
| 🟩 초록 | `pad_C` |
| 🟦 파랑 | `pad_D` |
| 🟥 빨강 | `pad_B` |
| 🟨 노랑 | `pad_E` |

알맞은 패드에 배송하면 큐브가 사라지고 벨트가 새 큐브를 공급합니다.

> ⚠️ **잘못된 패드에 배송하면 벤치마크가 종료됩니다.** 이후 `pick_entity`는 모두 `benchmark terminated: wrong_color_destination` 오류를 냅니다. 다음 명령으로 다시 활성화할 수 있습니다.
> ```python
> await session.invoke("configure_benchmark", {"benchmark": {}})
> ```

### 새로운 스킬 두 가지

| 스킬 | 역할 | 사전 조건 |
|---|---|---|
| `pick_entity` | 큐브 집기 | 로봇이 큐브 가까이(약 1m), 손은 비어 있어야 함 |
| `place_entity` | 들고 있는 큐브를 패드에 배송 | 물체를 들고 있어야 함 |

둘 다 `go_to`와 같은 target 형식을 사용합니다. 정확한 이름으로 특정 큐브를 집거나 특별한 이름 `"cube"`로 가장 가까운 큐브를 집을 수 있습니다. 가까운 큐브가 둘이면 엉뚱한 큐브를 집을 수 있으므로 먼저 원하는 큐브 가까이 이동한 뒤 집으세요.

## 1단계: 장면 조사하기

현재 모든 물체의 위치를 확인합니다. 앞선 섹션에서 로봇이 움직였고 컨베이어도 계속 큐브를 공급했으므로 `scene_state`에서 정확한 위치를 읽고 POV 스크린샷도 촬영합니다.

장면 상태는 실제 로봇에서는 읽을 수 없습니다. 여기서는 코드의 동작을 단순하게 설명하기 위해 먼저 사용합니다.

In [ ]:
# 로봇과 큐브의 위치를 확인합니다
scene = await session.state.get("scene_state")

robot = scene.entities["robot"]
rp = robot.pose.position
print(f"Robot at ({rp[0]:+.2f}, {rp[1]:+.2f}), yaw={robot.pose.yaw_deg:.0f}°")

print("\nCubes (sorted by distance from the robot):")
cubes = []
for eid, e in scene.entities.items():
    if eid.startswith("cube_") and e.visible:
        p = e.pose.position
        dist = ((p[0] - rp[0]) ** 2 + (p[1] - rp[1]) ** 2) ** 0.5
        color = e.state.get("color", "?") if e.state else "?"
        cubes.append((dist, eid, color, p))

for dist, eid, color, p in sorted(cubes):
    print(f"  {eid:8s} color={color:6s} at ({p[0]:+.2f}, {p[1]:+.2f})  -> {dist:.2f} m away")

await screenshot("\nRobot's POV:")

## 2단계: 손이 닿는 거리까지 이동하기

`pick_entity`는 로봇이 큐브에 충분히 가까울 때만 작동합니다. 조사에서 찾은 가장 가까운 큐브 옆으로 `go_to`를 사용해 이동합니다.

> **자주 빠지는 함정:** 로봇은 컨베이어 바로 앞에 생성되지 않고 첫 상자에서 약간 옆에 있어 손이 닿지 않습니다. 화면상 가까워 보여도 실제 집기 범위 안이라는 뜻은 아닙니다. 눈대중 대신 먼저 이동하고 `scene_state`로 거리를 측정하세요.

In [ ]:
# 손이 닿도록 가장 가까운 큐브까지 이동합니다
nearest_dist, nearest_cube, _, _ = min(cubes)
print(f"Nearest cube: {nearest_cube} ({nearest_dist:.2f} m away) — walking to it...")

result = await session.invoke(
    "go_to", {"target": {"kind": "entity", "entity_id": nearest_cube}}, timeout_s=300
)
print(f"go_to {nearest_cube} -> {result.status}")

# 현재 거리를 확인합니다
scene = await session.state.get("scene_state")
rp = scene.entities["robot"].pose.position
cp = scene.entities[nearest_cube].pose.position
dist = ((cp[0] - rp[0]) ** 2 + (cp[1] - rp[1]) ** 2) ** 0.5
print(f"Distance to {nearest_cube} is now {dist:.2f} m")

await screenshot("In position:")

## 3단계: 집기

큐브 옆에 섰으므로 `"cube"`, 즉 가장 가까운 큐브를 집고 장면 상태에서 실제로 무엇을 들었는지 확인합니다. 들린 큐브는 `attached_to`가 설정되고 로봇 상태는 `holding`이 됩니다.

컨베이어는 계속 움직이므로 목표 큐브 쪽으로 걸어가도 옆 큐브를 집을 수 있습니다. 다음 코드는 실제로 집은 큐브의 색상을 읽어 알맞은 패드로 보냅니다. 반드시 특정 큐브가 필요하다면 정확한 `entity_id`로 집으세요.

In [ ]:
# 가장 가까운 큐브를 집습니다 ("cube" = 가장 가까운 큐브)
result = await session.invoke(
    "pick_entity", {"target": {"kind": "entity", "entity_id": "cube"}}, timeout_s=300
)
print(f"pick_entity -> {result.status}")

# 장면 상태에서 어떤 큐브가 로봇에 연결되었는지 확인합니다
scene = await session.state.get("scene_state")
held = [
    (eid, e.state.get("color", "?") if e.state else "?")
    for eid, e in scene.entities.items()
    if eid.startswith("cube_") and e.attached_to
]
print(f"Now holding: {held}")

state = await session.state.get("robot_status")
print(f"Robot status: {state.robot.status}")   # should say 'holding'

await screenshot("After picking:")

## 4단계: 알맞은 패드로 옮겨 배송하기

로봇은 걷는 동안 큐브를 자동으로 들고 갑니다. 잘못된 패드에 놓으면 벤치마크가 끝나므로 아래 코드는 다음을 수행합니다.

1. 실제로 들고 있는 큐브의 **색상** 확인
2. 딕셔너리에서 알맞은 패드 찾기
3. `go_to`로 해당 패드까지 이동
4. `place_entity`로 배송
5. 배송 검증: 성공하면 큐브의 `visible`이 `False`가 되고 로봇은 `ready`로 돌아감

성공한 큐브는 사라지므로 세계 상태를 읽어야 확실히 확인할 수 있습니다. `place_entity`의 target을 생략하면 가장 가까운 구역에 놓지만, 색상 규칙에서 사고를 피하기 위해 항상 패드를 명시합니다.

In [ ]:
# 뷰어의 표지판을 보고 패드별 허용 색상을 확인합니다
COLOR_TO_PAD = {"green": "pad_C", "blue": "pad_D", "red": "pad_B", "yellow": "pad_E"}

# 1-2. 무엇을 들고 있으며 어디로 가져가야 하는지 확인합니다
scene = await session.state.get("scene_state")
held_cube, held_color = next(
    (eid, e.state.get("color"))
    for eid, e in scene.entities.items()
    if eid.startswith("cube_") and e.attached_to
)
target_pad = COLOR_TO_PAD[held_color]
print(f"Holding {held_cube} ({held_color}) -> delivering to {target_pad}")

# 3. 경로 계획기를 사용해 이동합니다
result = await session.invoke(
    "go_to", {"target": {"kind": "entity", "entity_id": target_pad}}, timeout_s=300
)
print(f"go_to {target_pad} -> {result.status}")

# 4. 배송합니다
result = await session.invoke(
    "place_entity", {"target": {"kind": "entity", "entity_id": target_pad}}, timeout_s=300
)
print(f"place_entity -> {result.status}")

# 5. 눈이 아닌 데이터로 검증합니다. 배송한 큐브는 사라져야 하며
#    로봇은 더 이상 아무것도 들고 있지 않아야 합니다.
scene = await session.state.get("scene_state")
state = await session.state.get("robot_status")
print(f"{held_cube} still visible? {scene.entities[held_cube].visible}")
print(f"Robot status: {state.robot.status}")          # back to 'ready'
delivered = not scene.entities[held_cube].visible and state.robot.status != "holding"
print(f"배송합니다y confirmed: {delivered}")

await screenshot("After delivering:")

### 생각해 볼 질문

1. 모든 큐브에서 멀리 떨어진 상태로 `pick_entity`를 호출하면 어떻게 될까요? 예측한 뒤 빈 공간에서 실행하고 `result`를 확인하세요.
2. 이미 큐브를 들고 있는데 다시 `pick_entity`를 호출하면 어떻게 될까요?
3. 색상이 맞지 않는 패드에 배송하면 어떻게 될까요? 위의 경고를 먼저 읽으세요.
4. 배송 후 뷰어만 보거나 `result.status`만 믿지 않고 `scene_state`의 `visible`과 로봇 상태를 확인하는 것이 왜 더 좋은 습관일까요?

## 연습 문제: 빨간 큐브 배송하기

컨베이어에서 **빨간 큐브**를 찾아 알맞은 패드로 배송하는 프로그램을 작성하세요.

- 큐브 이름을 하드코딩하지 말고 `scene_state`로 빨간 큐브를 찾으세요. 벨트가 움직이고 다시 채워져 실행마다 ID가 달라집니다.
- 큐브까지 이동해 집은 뒤 **실제로 들고 있는 물체**가 빨간색인지 확인하세요.
- 표를 보고 올바른 패드로 배송하세요. 잘못 선택하면 벤치마크가 끝납니다.
- 배송이 받아들여졌는지 **코드로 검증**하세요. 배송된 큐브는 어떻게 되나요?

두 큐브 사이에 멈추면 `"cube"`가 원하는 큐브가 아닐 수 있습니다. 코드가 이를 어떻게 알아차릴 수 있는지, 더 안전한 집기 방법은 무엇인지 생각해 보세요.

In [ ]:
# ============================================================
# 여기에 코드를 작성하세요
#
# 계획:
#   1. scene_state를 읽고 빨간 큐브를 찾습니다 (check e.state["color"])
#   2. go_to로 해당 큐브까지 이동합니다
#   3. pick_entity — 정확한 entity_id를 사용합니다("cube"보다 안전함)
#   4. attached_to로 실제 들고 있는 물체가 빨간 큐브인지 확인합니다
#   5. 빨간색 패드로 이동해 place_entity를 호출합니다
#   6. scene_state를 다시 읽어 배송 성공을 검증합니다.
#      들었던 큐브의 visible이 False여야 합니다
#
# 유용한 코드 조각:
#   await session.invoke("go_to", {"target": {"kind": "entity", "entity_id": "..."}}, timeout_s=300)
#   color = e.state.get("color") if e.state else None
# ============================================================



---
## 정리

In [ ]:
await session.disconnect()
await client.robots.delete(robot_id)
print("Robot deleted.")